# Phase 05 · Bài 03 — Word Embeddings và Word2Vec

**Mục tiêu:** hiểu distributional hypothesis, tạo cặp Skip-gram, giải thích hai bảng vector và nắm vai trò của negative sampling.

## 1. Từ đếm từ sang hình học vector

Bag of Words biết hai tài liệu có chung từ, nhưng không tự biết “dog” và “puppy” có thể gần nghĩa. Word embedding gán mỗi từ một vector d chiều. Thay vì một chiều độc lập cho mỗi từ, mô hình học tọa độ sao cho quan hệ ngữ cảnh được thể hiện qua hình học.

Giả thuyết phân bố thường được tóm tắt: từ xuất hiện trong ngữ cảnh tương tự thường có nghĩa tương tự. Đây là xu hướng thống kê chứ không phải định luật tuyệt đối. Từ trái nghĩa có thể xuất hiện trong cùng ngữ cảnh; từ phổ biến có thể gần nhiều từ; chất lượng vector phụ thuộc corpus, preprocessing, kích thước cửa sổ và cách học.

## 2. Skip-gram tạo dữ liệu giám sát từ văn bản

Với mỗi vị trí, Skip-gram lấy một token làm từ trung tâm và các token quanh nó làm ngữ cảnh. Nếu cửa sổ là 2, mô hình tạo các cặp center/context ở tối đa hai vị trí hai bên, không ghép token với chính nó. Một câu dài tạo nhiều cặp; tăng cửa sổ thu thêm ngữ cảnh xa nhưng làm tín hiệu ít cục bộ hơn.

CBOW làm hướng ngược lại: lấy các từ ngữ cảnh để dự đoán từ trung tâm. Skip-gram thường cung cấp nhiều cặp học quanh từ hiếm; CBOW gộp các ngữ cảnh nên có thể nhanh hơn. Cả hai học biểu diễn tĩnh, nhưng khác mục tiêu dự đoán.

## 3. Hai ma trận và phép tra cứu embedding

Word2Vec có bảng vector cho từ trung tâm W và bảng vector ngữ cảnh W'. Có thể hình dung one-hot của từ trung tâm chọn một hàng của W; tích vô hướng giữa hàng đó với hàng context trong W' tạo score cho cặp. Trong triển khai thực tế không cần lưu một ma trận one-hot khổng lồ: index tra trực tiếp hàng tương ứng.

Nếu dùng softmax đầy đủ, mỗi cặp cần chuẩn hóa score trên toàn vocabulary. Khi vocabulary lớn, điều này tốn kém. Negative sampling biến tác vụ thành phân biệt cặp thật với vài cặp giả lấy mẫu.

## 4. Negative sampling và objective

Gọi v_c là vector center, u_o là vector context thật, u_k là vector của negative sample thứ k, và σ là sigmoid. Một dạng loss cho một positive pair là:

L = -log σ(v_c · u_o) - Σ_k log σ(-v_c · u_k)

Thành phần thứ nhất tăng xác suất cặp quan sát được; phần thứ hai giảm xác suất các cặp giả. Gradient descent cập nhật cả vector center lẫn các vector context liên quan. Số lượng và phân phối negative samples ảnh hưởng đến tốc độ, chất lượng và bias của vector. Bài thực hành dùng hai negative sample lấy đều trong vocabulary để code gọn; các hệ thống Word2Vec cổ điển thường dùng phân phối đã điều chỉnh theo tần suất.

## 5. Đọc không gian và hiểu giới hạn

Cosine similarity đo góc giữa vector, thường dùng để tìm hàng xóm. Các phép toán tương tự kiểu “king − man + woman” là quan sát hình học trên corpus, không có nghĩa mô hình đã mã hóa một quy tắc logic ổn định. Trên corpus nhỏ, hàng xóm có thể phản ánh câu mẫu hoặc seed ngẫu nhiên nhiều hơn nghĩa rộng.

Word2Vec có một vector cho mỗi loại từ, nên “bank” trong financial bank và river bank dùng chung vector. Từ chưa xuất hiện trong training thường không có vector. Contextual embeddings xử lý nghĩa theo từng lần xuất hiện; FastText ở bài 04 dùng n-gram để cải thiện từ hiếm/OOV theo hướng khác.

Notebook thực hành huấn luyện Skip-gram nhỏ trên corpus đồ uống, in vector và cosine, rồi so số cặp ở các kích thước window. Đây là cách quan sát cơ chế học; kết quả không phải phép đo chất lượng trên ngôn ngữ tự nhiên nói chung.

## A. Chạy code mẫu của bài

Bài dùng NumPy và seed cố định để có thể tái lập toy run.

In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "03-word-embeddings-word2vec"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "Word2Vec from Scratch")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: Word2Vec from Scratch
Lệnh: phases\05-nlp-foundations-to-advanced\03-word-embeddings-word2vec\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
nearest to cat:
  dog          0.988
  mouse        0.750
  mat          0.622
  rug          0.598

nearest to dog:
  cat          0.988
  mouse        0.703
  mat          0.566
  rug          0.537

nearest to sat:
  rug          0.795
  mat          0.762
  on           0.616
  the          0.594

nearest to chased:
  a            0.977
  mouse        0.823
  cat          0.473
  dog          0.393


## B. Thực hành riêng — huấn luyện Skip-gram nhỏ

Ta huấn luyện vector 8 chiều trên corpus ngắn. Mỗi positive context đi cùng hai negative samples. Loss logistic kéo điểm cặp thật lên và điểm cặp giả xuống.

**Giới hạn:** corpus ngắn làm vector dao động. Dùng kết quả để kiểm tra luồng cập nhật, không suy ra quan hệ ngữ nghĩa tổng quát từ một toy run.

In [2]:
import re
import numpy as np

sentences = [
    "coffee tastes rich", "coffee with milk", "tea tastes light",
    "tea with lemon", "coffee tastes good", "tea tastes good",
] * 8
docs = [re.findall(r"[a-z]+", sentence.lower()) for sentence in sentences]
vocab = {word: index for index, word in enumerate(sorted({w for doc in docs for w in doc}))}
id_to_word = {index: word for word, index in vocab.items()}

def make_pairs(documents, window=1):
    pairs = []
    for doc in documents:
        for center_pos, center in enumerate(doc):
            start = max(0, center_pos - window)
            stop = min(len(doc), center_pos + window + 1)
            for context_pos in range(start, stop):
                if context_pos != center_pos:
                    pairs.append((vocab[center], vocab[doc[context_pos]]))
    return pairs

def sigmoid(value):
    value = np.clip(value, -20.0, 20.0)
    return 1.0 / (1.0 + np.exp(-value))

pairs = make_pairs(docs, window=1)
rng = np.random.default_rng(7)
dim, learning_rate, epochs = 8, 0.05, 120
W = rng.normal(0.0, 0.1, (len(vocab), dim))
C = rng.normal(0.0, 0.1, (len(vocab), dim))
negative_pool = np.arange(len(vocab))

for epoch in range(epochs):
    for pair_index in rng.permutation(len(pairs)):
        center, positive = pairs[pair_index]
        negatives = rng.choice(negative_pool[negative_pool != positive], size=2, replace=True)
        # Copy before updates so both gradients use the same parameter values.
        v, u = W[center].copy(), C[positive].copy()
        error = sigmoid(v @ u) - 1.0
        W[center] -= learning_rate * error * u
        C[positive] -= learning_rate * error * v
        for negative in negatives:
            v, u_neg = W[center].copy(), C[negative].copy()
            error_neg = sigmoid(v @ u_neg)
            W[center] -= learning_rate * error_neg * u_neg
            C[negative] -= learning_rate * error_neg * v

def cosine_words(left, right):
    a, b = W[vocab[left]], W[vocab[right]]
    return float((a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

print("vocabulary:", list(vocab))
print("positive skip-gram pairs:", len(pairs))
print("embedding matrix shape:", W.shape)
for word in ("coffee", "tea", "tastes"):
    print(f"{word:8s} ->", np.round(W[vocab[word]][:4], 3))
print("cosine(coffee, tea):", round(cosine_words("coffee", "tea"), 3))
print("cosine(coffee, lemon):", round(cosine_words("coffee", "lemon"), 3))
print("\nCosine phụ thuộc seed, corpus, window và số epoch.")

vocabulary: ['coffee', 'good', 'lemon', 'light', 'milk', 'rich', 'tastes', 'tea', 'with']
positive skip-gram pairs: 192
embedding matrix shape: (9, 8)
coffee   -> [ 0.492 -0.218  0.486 -1.574]
tea      -> [ 0.721 -0.226  0.522 -1.566]
tastes   -> [-0.55   1.535  0.886 -0.082]
cosine(coffee, tea): 0.993
cosine(coffee, lemon): 0.795

Cosine phụ thuộc seed, corpus, window và số epoch.


## C. Thí nghiệm — thay cửa sổ ngữ cảnh

Trước khi huấn luyện lại, đếm số positive pairs với các cửa sổ khác nhau. Cửa sổ lớn đưa thêm ngữ cảnh xa vào tín hiệu học.

In [3]:
probe_doc = ["coffee", "tastes", "rich", "coffee", "tastes", "rich"]
for window in (1, 2, 3):
    pairs_for_window = make_pairs([probe_doc], window=window)
    print(f"window={window}: {len(pairs_for_window)} pairs")

print("Ví dụ:", [(id_to_word[c], id_to_word[x]) for c, x in make_pairs(docs[:1], 1)[:8]])

window=1: 10 pairs
window=2: 18 pairs
window=3: 24 pairs
Ví dụ: [('coffee', 'tastes'), ('tastes', 'coffee'), ('tastes', 'rich'), ('rich', 'tastes')]


## D. Tự kiểm tra

1. Tại sao Skip-gram có bảng vector cho từ trung tâm và từ ngữ cảnh?
2. Positive và negative samples đóng vai trò gì trong loss?
3. Vì sao cosine cao trên corpus nhỏ chưa đủ khẳng định hai từ đồng nghĩa?
4. Word2Vec xử lý từ chưa gặp trong training ra sao? So sánh với FastText ở bài 04.